<a href="https://colab.research.google.com/github/Saumya731/AI-Research-Power-Intelligent-System/blob/main/Project__2_AI_Research_Power_Intelligent_System_project_in_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q transformers
!pip install -q pandas
!pip install -q numpy
!pip install -q torch
!pip install -q scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import torch
import faiss

from datasets import load_dataset

from sentence_transformers import SentenceTransformer

from transformers import pipeline

from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
dataset = load_dataset(
    "CShorten/ML-ArXiv-Papers",
    split="train"
)

print(dataset)

README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

ML-Arxiv-Papers.csv:   0%|          | 0.00/147M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/117592 [00:00<?, ? examples/s]

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [ ]:
dataset = load_dataset(
    "CShorten/ML-ArXiv-Papers",
    split="train"
)

print(dataset)

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [ ]:
df = dataset.to_pandas()

df.head()

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


In [ ]:
df = df[['title','abstract']]

df.dropna(inplace=True)

df.reset_index(drop=True,inplace=True)

print(df.shape)

df.head()

(117592, 2)


,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


In [ ]:
for i in range(5):

    print("="*100)

    print("Title:\n")

    print(df.iloc[i]['title'])

    print()

    print("Abstract:\n")

    print(df.iloc[i]['abstract'][:500])

Title:

Learning from compressed observations

Abstract:

  The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part 
Title:

Sensor Networks with Random Links: Topology Design for Distributed
  Consensus

Abstract:

  In a sensor network, in practice, the communication among sensors is subject
to:(1) errors or failures at random times; (3) costs; and(2) constraints since
sensors and networks operate under scarce resources, such as power, data rate,
or communication. The signal-to-noise ratio (SNR) is usually a main factor in
determining the probability

In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Model Loaded Successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded Successfully


In [ ]:
papers = df['abstract'].tolist()[:5000]

embeddings = model.encode(
    papers,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(embeddings.shape)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

(5000, 384)


In [ ]:
embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)

print(embeddings.shape)

(5000, 384)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS Index Created Successfully")

print(index.ntotal)

FAISS Index Created Successfully
5000


In [ ]:
faiss.write_index(
    index,
    "research_index.faiss"
)

print("Index Saved")

Index Saved


In [ ]:
def search_paper(query, top_k=5):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype(np.float32)

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    for rank, idx in enumerate(indices[0]):

        print("="*100)

        print("Rank:", rank+1)

        print()

        print("Title:\n")

        print(df.iloc[idx]["title"])

        print()

        print("Abstract:\n")

        print(df.iloc[idx]["abstract"][:700])

        print()

        print("Distance:", distances[0][rank])

In [ ]:
search_paper("Deep Learning for Image Classification")

Rank: 1

Title:

Unsupervised feature learning by augmenting single images

Abstract:

  When deep learning is applied to visual object recognition, data augmentation
is often used to generate additional training data without extra labeling cost.
It helps to reduce overfitting and increase the performance of the algorithm.
In this paper we investigate if it is possible to use data augmentation as the
main component of an unsupervised feature learning architecture. To that end we
sample a set of random image patches and declare each of them to be a separate
single-image surrogate class. We then extend these trivial one-element classes
by applying a variety of transformations to the initial 'seed' patches. Finally
we train a convolutional neural network to discriminate between t

Distance: 0.8688776
Rank: 2

Title:

Deeply Coupled Auto-encoder Networks for Cross-view Classification

Abstract:

  The comparison of heterogeneous samples extensively exists in many
applications, especially i

In [ ]:
import transformers
print(transformers.__version__)

5.12.1


In [ ]:
!pip uninstall -y transformers tokenizers
!pip install -q transformers==4.41.2
!pip install -q accelerate
!pip install -q sentencepiece

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1


In [ ]:
def recommend(title):

    index_title = df[df["title"]==title].index[0]

    query = embeddings[index_title].reshape(1,-1)

    D,I = index.search(query,6)

    for idx in I[0][1:]:

        print("="*80)

        print(df.iloc[idx]["title"])

In [ ]:
def recommend(title):

    index_title = df[df["title"]==title].index[0]

    query = embeddings[index_title].reshape(1,-1)

    D,I = index.search(query,6)

    for idx in I[0][1:]:

        print("="*80)

        print(df.iloc[idx]["title"])

In [ ]:
recommend(df.iloc[100]["title"])

Copula-based Kernel Dependency Measures
Speedy Model Selection (SMS) for Copula Models
Inference, Sampling, and Learning in Copula Cumulative Distribution
  Networks
Flexible sampling of discrete data correlations without the marginal
  distributions
Inference-less Density Estimation using Copula Bayesian Networks


In [ ]:
!pip uninstall -y transformers tokenizers
!pip install -q transformers==4.41.2
!pip install -q accelerate sentencepiece

Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import transformers
print(transformers.__version__)

4.41.2


In [ ]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=device
)

print("✅ Summarizer Loaded Successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

✅ Summarizer Loaded Successfully!


In [ ]:
text = """
Artificial Intelligence is transforming healthcare by enabling disease prediction,
medical image analysis, and personalized treatment recommendations.
Deep learning models help doctors make faster and more accurate decisions.
"""

summary = summarizer(
    text,
    max_length=50,
    min_length=20,
    do_sample=False
)

print(summary[0]["summary_text"])

Your max_length is set to 50, but your input_length is only 39. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=19)


 Artificial Intelligence is transforming healthcare by enabling disease prediction, medical image analysis, and personalized treatment recommendations . Deep learning models help doctors make faster and more accurate decisions .
